# Schema Evolution
1. Adding New Columns (Manual / Automatic)
2. Widening Data Types (Supported Delta >= 3.2): Sometimes we need to expand a column's data type to accommodate larger values. Delta Lake allows "widening" type conversions that won't lose data, such as:
- `INT` to `BIGINT`
- `FLOAT` to `DOUBLE`
- `VARCHAR(10)` to `VARCHAR(20)`

3. Nested Structure Evolution (Manual / Automatic): Delta Lake supports evolution of complex data types like structs and arrays. We can:
- Add new fields to structs
- Modify nested field types
- Add new elements to arrays

4. Column Position Changes (Manual / Automatic): we can reorganize our columns

Note: 
- `INSERT` works by matching columns by position
- `MERGE` works by matching columns by name

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://warehouse/default/deltalake/invoices_se --recursive

In [2]:
%load_ext sql

In [3]:
%sql spark

#### Scenario 1: Adding New Columns 

In [6]:
%%sql

DROP TABLE IF EXISTS spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

++
||
++
++

In [7]:
%%sql

SELECT *
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`
WHERE customer_id BETWEEN 1 AND 5

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11
1,I178410,Male,61,Clothing,5,1500.4,Credit Card,2021-11-26,Metrocity,None
2,I158163,Male,34,Shoes,2,1200.34,Cash,2023-03-03,Kanyon,None
3,I262373,Male,44,Toys,3,107.52,Credit Card,2022-12-01,Cevahir AVM,None
4,I334895,Male,25,Food & Beverage,5,26.15,Cash,2021-08-15,Kanyon,None
5,I202043,Female,21,Toys,1,35.84,Credit Card,2021-07-25,Metrocity,None


In [8]:
%%sql

CREATE OR REPLACE TABLE spark_catalog.deltalake.invoices_se (
  customer_id INT NOT NULL,  
  invoice_no STRING,
  price FLOAT, 
  invoice_date DATE
)
USING DELTA; 

INSERT INTO spark_catalog.deltalake.invoices_se
SELECT customer_id, invoice_no, price, invoice_date
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`
WHERE customer_id BETWEEN 1 AND 5

Running query in 'SparkSession'

++
||
++
++

In [9]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3,Field 4
1,I178410,1500.4000244140625,2021-11-26
2,I158163,1200.3399658203125,2023-03-03
3,I262373,107.5199966430664,2022-12-01
4,I334895,26.149999618530273,2021-08-15
5,I202043,35.84000015258789,2021-07-25


In [10]:
%%sql

ALTER TABLE spark_catalog.deltalake.invoices_se
ADD COLUMNS (quantity INT);

Running query in 'SparkSession'

++
||
++
++

In [11]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se
SELECT customer_id, invoice_no, price, invoice_date, quantity
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`
WHERE customer_id BETWEEN 6 AND 10 

Running query in 'SparkSession'

++
||
++
++

In [12]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
1,I178410,1500.4000244140625,2021-11-26,None
2,I158163,1200.3399658203125,2023-03-03,None


In [13]:
%%sql

SET spark.databricks.delta.schema.autoMerge.enabled = false;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
spark.databricks.delta.schema.autoMerge.enabled,false


In [14]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se
SELECT customer_id, invoice_no, price, invoice_date, quantity, payment_method 
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`
WHERE customer_id BETWEEN 11 AND 15 

Running query in 'SparkSession'

RuntimeError: [_LEGACY_ERROR_TEMP_DELTA_0007] A schema mismatch detected when writing to the Delta table (Table ID: 5745df82-9f19-46e5-bfaf-47feb36dd8c7).
To enable schema migration using DataFrameWriter or DataStreamWriter, please set:
'.option("mergeSchema", "true")'.
For other operations, set the session configuration
spark.databricks.delta.schema.autoMerge.enabled to "true". See the documentation
specific to the operation for details.

Table schema:
root
-- customer_id: integer (nullable = false)
-- invoice_no: string (nullable = true)
-- price: float (nullable = true)
-- invoice_date: date (nullable = true)
-- quantity: integer (nullable = true)


Data schema:
root
-- customer_id: integer (nullable = true)
-- invoice_no: string (nullable = true)
-- price: float (nullable = true)
-- invoice_date: date (nullable = true)
-- quantity: integer (nullable = true)
-- payment_method: string (nullable = true)

         


In [15]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5
1,I178410,1500.4000244140625,2021-11-26,None
2,I158163,1200.3399658203125,2023-03-03,None


#### Scenario 2: Type Widening

In [16]:
%%sql

ALTER TABLE spark_catalog.deltalake.invoices_se 
SET TBLPROPERTIES ('delta.enableTypeWidening' = 'true');

Running query in 'SparkSession'

++
||
++
++

In [17]:
%%sql

DESCRIBE TABLE spark_catalog.deltalake.invoices_se; 

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3
customer_id,int,None
invoice_no,string,None
price,float,None
invoice_date,date,None
quantity,int,None


In [18]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se 
VALUES (123456789012345,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card')

Running query in 'SparkSession'

ArithmeticException: [CAST_OVERFLOW_IN_TABLE_INSERT] Fail to insert a value of "BIGINT" type into the "INT" type column `customer_id` due to an overflow. Use `try_cast` on the input value to tolerate overflow and return NULL instead.

In [19]:
%%sql

ALTER TABLE spark_catalog.deltalake.invoices_se
ALTER COLUMN customer_id TYPE BIGINT;

Running query in 'SparkSession'

RuntimeError: [DELTA_UNSUPPORTED_ALTER_TABLE_CHANGE_COL_OP] ALTER TABLE CHANGE COLUMN is not supported for changing column customer_id from INT NOT NULL to BIGINT NOT NULL


In [20]:
%%sql

DESCRIBE TABLE spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

5 rows affected.

Field 1,Field 2,Field 3
customer_id,int,None
invoice_no,string,None
price,float,None
invoice_date,date,None
quantity,int,None


In [21]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se 
VALUES (123456789012345,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card')

Running query in 'SparkSession'

ArithmeticException: [CAST_OVERFLOW_IN_TABLE_INSERT] Fail to insert a value of "BIGINT" type into the "INT" type column `customer_id` due to an overflow. Use `try_cast` on the input value to tolerate overflow and return NULL instead.

In [22]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se
WHERE customer_id = 123456789012345;

Running query in 'SparkSession'

++
||
++
++

#### Scenario 3: Nested Structure Evolution

In [ ]:
%%sql

ALTER TABLE spark_catalog.deltalake.invoices_se
ADD COLUMNS purchase_details STRUCT<
  mall_pin_code INT,
  store_code INT
>;

Running query in 'SparkSession'

++
||
++
++

In [24]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se 
VALUES (16,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', STRUCT(12345, 879));

Running query in 'SparkSession'

RuntimeError: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "col6" due to data type mismatch: cannot cast "STRING" to "STRUCT<mall_pin_code: INT, store_code: INT>".; line 1 pos 0;
'AppendData RelationV2[customer_id#4037, invoice_no#4038, price#4039, invoice_date#4040, quantity#4041, purchase_details#4042] spark_catalog.deltalake.invoices_se spark_catalog.deltalake.invoices_se, false
+- 'Project [col1#4043 AS customer_id#4050, col2#4044 AS invoice_no#4051, cast(col3#4045 as float) AS price#4052, cast(col4#4046 as date) AS invoice_date#4053, col5#4047 AS quantity#4054, cast(col6#4048 as struct<mall_pin_code:int,store_code:int>) AS purchase_details#4055, col7#4049]
   +- LocalRelation [col1#4043, col2#4044, col3#4045, col4#4046, col5#4047, col6#4048, col7#4049]



In [25]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6
1,I178410,1500.4000244140625,2021-11-26,None,None
2,I158163,1200.3399658203125,2023-03-03,None,None


In [26]:
%%sql

ALTER TABLE spark_catalog.deltalake.invoices_se
ALTER COLUMN purchase_details.mall_pin_code TYPE BIGINT;

Running query in 'SparkSession'

RuntimeError: [DELTA_UNSUPPORTED_ALTER_TABLE_CHANGE_COL_OP] ALTER TABLE CHANGE COLUMN is not supported for changing column purchase_details.mall_pin_code from INT to BIGINT


In [27]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se 
VALUES (17,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', STRUCT(123456789012346, 765));

Running query in 'SparkSession'

RuntimeError: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "col6" due to data type mismatch: cannot cast "STRING" to "STRUCT<mall_pin_code: INT, store_code: INT>".; line 1 pos 0;
'AppendData RelationV2[customer_id#5154, invoice_no#5155, price#5156, invoice_date#5157, quantity#5158, purchase_details#5159] spark_catalog.deltalake.invoices_se spark_catalog.deltalake.invoices_se, false
+- 'Project [col1#5160 AS customer_id#5167, col2#5161 AS invoice_no#5168, cast(col3#5162 as float) AS price#5169, cast(col4#5163 as date) AS invoice_date#5170, col5#5164 AS quantity#5171, cast(col6#5165 as struct<mall_pin_code:int,store_code:int>) AS purchase_details#5172, col7#5166]
   +- LocalRelation [col1#5160, col2#5161, col3#5162, col4#5163, col5#5164, col6#5165, col7#5166]



In [28]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6
1,I178410,1500.4000244140625,2021-11-26,None,None
2,I158163,1200.3399658203125,2023-03-03,None,None


In [29]:
%%sql

ALTER TABLE spark_catalog.deltalake.invoices_se
ADD COLUMN purchase_details.store_loc STRING;

Running query in 'SparkSession'

++
||
++
++

In [30]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se 
VALUES (17,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', STRUCT(7612, 765, 'ground floor'));

Running query in 'SparkSession'

RuntimeError: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "col6" due to data type mismatch: cannot cast "STRING" to "STRUCT<mall_pin_code: INT, store_code: INT, store_loc: STRING>".; line 1 pos 0;
'AppendData RelationV2[customer_id#5451, invoice_no#5452, price#5453, invoice_date#5454, quantity#5455, purchase_details#5456] spark_catalog.deltalake.invoices_se spark_catalog.deltalake.invoices_se, false
+- 'Project [col1#5457 AS customer_id#5464, col2#5458 AS invoice_no#5465, cast(col3#5459 as float) AS price#5466, cast(col4#5460 as date) AS invoice_date#5467, col5#5461 AS quantity#5468, cast(col6#5462 as struct<mall_pin_code:int,store_code:int,store_loc:string>) AS purchase_details#5469, col7#5463]
   +- LocalRelation [col1#5457, col2#5458, col3#5459, col4#5460, col5#5461, col6#5462, col7#5463]



In [31]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6
1,I178410,1500.4000244140625,2021-11-26,None,None
2,I158163,1200.3399658203125,2023-03-03,None,None


In [32]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se 
VALUES (21,	'I106485',	30.299999237060547,	'2022-12-01',	2,	'Debit Card', 
  NAMED_STRUCT(
    'mall_pin_code', 7612, 
    'store_code', 765, 
    'store_loc', 'ground floor', 
    'staff_id', 'ST12736'
  )
);

Running query in 'SparkSession'

RuntimeError: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "col6" due to data type mismatch: cannot cast "STRING" to "STRUCT<mall_pin_code: INT, store_code: INT, store_loc: STRING>".; line 1 pos 0;
'AppendData RelationV2[customer_id#6550, invoice_no#6551, price#6552, invoice_date#6553, quantity#6554, purchase_details#6555] spark_catalog.deltalake.invoices_se spark_catalog.deltalake.invoices_se, false
+- 'Project [col1#6556 AS customer_id#6563, col2#6557 AS invoice_no#6564, cast(col3#6558 as float) AS price#6565, cast(col4#6559 as date) AS invoice_date#6566, col5#6560 AS quantity#6567, cast(col6#6561 as struct<mall_pin_code:int,store_code:int,store_loc:string>) AS purchase_details#6568, col7#6562]
   +- LocalRelation [col1#6556, col2#6557, col3#6558, col4#6559, col5#6560, col6#6561, col7#6562]



In [33]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6
1,I178410,1500.4000244140625,2021-11-26,None,None
2,I158163,1200.3399658203125,2023-03-03,None,None


#### Scenario 4: Column Position Changes

In [34]:
%%sql

SET spark.databricks.delta.schema.autoMerge.enabled=false;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
spark.databricks.delta.schema.autoMerge.enabled,false


In [35]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6
1,I178410,1500.4000244140625,2021-11-26,None,None
2,I158163,1200.3399658203125,2023-03-03,None,None


In [36]:
%%sql

-- ALTER TABLE spark_catalog.deltalake.invoices_se ADD COLUMNS (age INT FIRST);
ALTER TABLE spark_catalog.deltalake.invoices_se ADD COLUMNS (age INT AFTER price)

Running query in 'SparkSession'

++
||
++
++

In [37]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se
SELECT customer_id, invoice_no, price, age, invoice_date, quantity, payment_method, NULL AS purchase_details
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`
WHERE customer_id BETWEEN 50 AND 55

Running query in 'SparkSession'

RuntimeError: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "payment_method" due to data type mismatch: cannot cast "STRING" to "STRUCT<mall_pin_code: INT, store_code: INT, store_loc: STRING>".; line 1 pos 0;
'AppendData RelationV2[customer_id#7120, invoice_no#7121, price#7122, age#7123, invoice_date#7124, quantity#7125, purchase_details#7126] spark_catalog.deltalake.invoices_se spark_catalog.deltalake.invoices_se, false
+- 'Project [customer_id#7127 AS customer_id#7139, invoice_no#7128 AS invoice_no#7140, cast(price#7133 as float) AS price#7141, age#7130 AS age#7142, invoice_date#7135 AS invoice_date#7143, quantity#7132 AS quantity#7144, cast(payment_method#7134 as struct<mall_pin_code:int,store_code:int,store_loc:string>) AS purchase_details#7145, purchase_details#7119]
   +- Project [customer_id#7127, invoice_no#7128, price#7133, age#7130, invoice_date#7135, quantity#7132, payment_method#7134, null AS purchase_details#7119]
      +- Filter ((customer_id#7127 >= 50) AND 

In [38]:
%%sql

SET spark.databricks.delta.schema.autoMerge.enabled = true;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2
spark.databricks.delta.schema.autoMerge.enabled,true


In [39]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

10 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7
1,I178410,1500.4000244140625,None,2021-11-26,None,None
2,I158163,1200.3399658203125,None,2023-03-03,None,None


In [40]:
%%sql

INSERT INTO spark_catalog.deltalake.invoices_se
SELECT customer_id, invoice_no, price, age, invoice_date, quantity, payment_method, category, NULL AS purchase_details
FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`
WHERE customer_id BETWEEN 56 AND 60

Running query in 'SparkSession'

RuntimeError: [DATATYPE_MISMATCH.CAST_WITHOUT_SUGGESTION] Cannot resolve "payment_method" due to data type mismatch: cannot cast "STRING" to "STRUCT<mall_pin_code: INT, store_code: INT, store_loc: STRING>".; line 1 pos 0;
'AppendData RelationV2[customer_id#8247, invoice_no#8248, price#8249, age#8250, invoice_date#8251, quantity#8252, purchase_details#8253] spark_catalog.deltalake.invoices_se spark_catalog.deltalake.invoices_se, false
+- 'Project [customer_id#8254 AS customer_id#8266, invoice_no#8255 AS invoice_no#8267, cast(price#8260 as float) AS price#8268, age#8257 AS age#8269, invoice_date#8262 AS invoice_date#8270, quantity#8259 AS quantity#8271, cast(payment_method#8261 as struct<mall_pin_code:int,store_code:int,store_loc:string>) AS purchase_details#8272, category#8258, purchase_details#8246]
   +- Project [customer_id#8254, invoice_no#8255, price#8260, age#8257, invoice_date#8262, quantity#8259, payment_method#8261, category#8258, null AS purchase_details#8246]
      +- Filter 

In [41]:
%%sql

MERGE INTO spark_catalog.deltalake.invoices_se tgt
USING (
  SELECT customer_id, invoice_no, price, age, invoice_date, quantity, payment_method, category, NULL AS purchase_details
  FROM PARQUET.`file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet`
  WHERE customer_id BETWEEN 56 AND 60
) src 
ON tgt.customer_id = src.customer_id
WHEN NOT MATCHED THEN
  INSERT *

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4
5,0,0,5


In [42]:
%%sql

SELECT * FROM spark_catalog.deltalake.invoices_se;

Running query in 'SparkSession'

15 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9
1,I178410,1500.4000244140625,None,2021-11-26,None,None,None,None
2,I158163,1200.3399658203125,None,2023-03-03,None,None,None,None


In [43]:

from pyspark.sql.functions import *
df = (
  spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
  .filter(col("customer_id").between(1, 10))
  .select("customer_id", "price", "invoice_date")
)
df.write.saveAsTable("spark_catalog.deltalake.invoices_se_spark_df")

In [44]:

df = (
  spark.read.parquet("file:///home/brijeshdhaker/IdeaProjects/bd-notebooks-module/data/invoices_1_100.parquet")
  .filter(col("customer_id").between(11, 25))
  .select("customer_id", "price", "invoice_date", "quantity", "payment_method")
)
df.write.mode("append").option("mergeSchema", "true").saveAsTable("spark_catalog.deltalake.invoices_se_spark_df")

AnalysisException: The column number of the existing table spark_catalog.deltalake.invoices_se_spark_df (struct<customer_id:int,price:double,invoice_date:date>) doesn't match the data schema (struct<customer_id:int,price:double,invoice_date:date,quantity:int,payment_method:string>).